# Apache Airflow — First Contact

Apache Airflow is a workflow orchestrator. Its job is not to do heavy business logic itself, but to **coordinate** tasks, schedule them, track their state, retry them when needed, and show you what happened. Think of it as the control tower for data pipelines.

Workflow orchestration means defining **what runs, when it runs, and in what order**. Execution is what each task actually does: query a database, transform records, write a report, call an API, or send a notification. Airflow specializes in the orchestration layer so engineers can manage repeatable, observable pipelines instead of running scripts by hand.

A DAG is a **Directed Acyclic Graph**: a set of tasks with one-way dependencies and no loops. In Citi terms: **Every morning at 06:00, Citi ops need an alert summary: how many HIGH/CRITICAL alerts per region overnight. This is a scheduled pipeline — exactly what Airflow orchestrates.**

```text
[Scheduler] → [DAG: extract_alerts → transform_summary → load_report] → [Postgres]
```

In [1]:
%pip install apache-airflow requests psycopg2-binary

import requests
import json
import psycopg2
from datetime import datetime, date
import time

Note: you may need to restart the kernel to use updated packages.


Verify Airflow is up before we interact with it

In [2]:
health_url = "http://localhost:8082/health"
response = requests.get(health_url, auth=("admin", "admin"), timeout=30)

print("Status code:", response.status_code)
try:
    payload = response.json()
    print(json.dumps(payload, indent=2))
except Exception:
    print(response.text)
    raise

if response.status_code == 200:
    print("Airflow is healthy")
else:
    raise RuntimeError(f"Airflow health check failed: {response.status_code}\n{response.text}")

Status code: 200
{
  "dag_processor": {
    "latest_dag_processor_heartbeat": null,
    "status": null
  },
  "metadatabase": {
    "status": "healthy"
  },
  "scheduler": {
    "latest_scheduler_heartbeat": "2026-03-31T23:01:11.879328+00:00",
    "status": "healthy"
  },
  "triggerer": {
    "latest_triggerer_heartbeat": "2026-03-31T23:01:07.679222+00:00",
    "status": "healthy"
  }
}
Airflow is healthy


In [3]:
USE_DOCKER_CLI = False


## Writing the DAG

A DAG file is a normal Python script placed in the `dags/` folder that Airflow watches.

Airflow's scheduler scans the `dags/` directory every ~30 seconds and picks up new DAG files automatically.

This notebook writes the DAG file content and shows you exactly what to place in the dags folder.

In [4]:
DAG_CONTENT = "from airflow import DAG\nfrom airflow.decorators import task\nfrom datetime import datetime\nimport psycopg2\n\n\nwith DAG(\n    dag_id=\"citi_alert_summary\",\n    schedule=\"0 6 * * *\",\n    start_date=datetime(2026, 1, 1),\n    catchup=False,\n    tags=[\"citi\", \"telemetry\", \"daily\"],\n) as dag:\n\n    @task\n    def extract_alerts():\n        conn = psycopg2.connect(\n            host=\"host.docker.internal\",\n            port=5432,\n            dbname=\"de_telemetry\",\n            user=\"de_admin\",\n            password=\"DeAdmin2026!\",\n        )\n        cur = conn.cursor()\n        cur.execute(\n            \"\"\"\n            SELECT alert_id, endpoint_id, severity, created_at\n            FROM alerts\n            WHERE created_at >= CURRENT_DATE - INTERVAL '1 day'\n            \"\"\"\n        )\n        rows = cur.fetchall()\n        colnames = [desc[0] for desc in cur.description]\n        cur.close()\n        conn.close()\n\n        results = []\n        for row in rows:\n            item = {}\n            for col, val in zip(colnames, row):\n                if col == \"created_at\" and val is not None:\n                    item[col] = val.isoformat()\n                else:\n                    item[col] = val\n            results.append(item)\n\n        print(f\"Extracted {len(results)} alerts\")\n        return results\n\n    @task\n    def transform_summary(alerts: list):\n        summary = {}\n        for alert in alerts:\n            severity = alert[\"severity\"]\n            summary[severity] = summary.get(severity, 0) + 1\n\n        print(summary)\n        return summary\n\n    @task\n    def load_report(summary: dict):\n        conn = psycopg2.connect(\n            host=\"host.docker.internal\",\n            port=5432,\n            dbname=\"de_telemetry\",\n            user=\"de_admin\",\n            password=\"DeAdmin2026!\",\n        )\n        cur = conn.cursor()\n        cur.execute(\n            \"\"\"\n            CREATE TABLE IF NOT EXISTS alert_daily_summary (\n                report_date DATE,\n                severity VARCHAR,\n                alert_count INT,\n                created_at TIMESTAMPTZ DEFAULT NOW()\n            )\n            \"\"\"\n        )\n\n        for severity, count in summary.items():\n            cur.execute(\n                \"\"\"\n                INSERT INTO alert_daily_summary (report_date, severity, alert_count)\n                VALUES (CURRENT_DATE, %s, %s)\n                \"\"\",\n                (severity, count),\n            )\n\n        conn.commit()\n        cur.close()\n        conn.close()\n\n        print(f\"Loaded {len(summary)} severity rows to alert_daily_summary\")\n\n    extracted = extract_alerts()\n    transformed = transform_summary(extracted)\n    load_report(transformed)\n"

print(DAG_CONTENT)

from airflow import DAG
from airflow.decorators import task
from datetime import datetime
import psycopg2


with DAG(
    dag_id="citi_alert_summary",
    schedule="0 6 * * *",
    start_date=datetime(2026, 1, 1),
    catchup=False,
    tags=["citi", "telemetry", "daily"],
) as dag:

    @task
    def extract_alerts():
        conn = psycopg2.connect(
            host="host.docker.internal",
            port=5432,
            dbname="de_telemetry",
            user="de_admin",
            password="DeAdmin2026!",
        )
        cur = conn.cursor()
        cur.execute(
            """
            SELECT alert_id, endpoint_id, severity, created_at
            FROM alerts
            WHERE created_at >= CURRENT_DATE - INTERVAL '1 day'
            """
        )
        rows = cur.fetchall()
        colnames = [desc[0] for desc in cur.description]
        cur.close()
        conn.close()

        results = []
        for row in rows:
            item = {}
            for col, val in zip(

## Where to Put the DAG File

The Airflow dags volume is mounted at `/opt/airflow/dags` **inside** the container.

You can copy the file into the container with:

```bash
docker cp citi_alert_summary.py citi_airflow:/opt/airflow/dags/
```

Or use the Airflow workflow from this notebook to write and deploy it.

Airflow rescans DAG files about every 30 seconds, so after copying the file, wait a bit before triggering it.

We write the DAG file directly into the Airflow container via docker exec

In [5]:
import subprocess
from pathlib import Path

import tempfile
local_dag_path = Path(tempfile.gettempdir()) / "citi_alert_summary.py"
local_dag_path.write_text(DAG_CONTENT, encoding="utf-8")

copy_cmd = ["docker", "cp", str(local_dag_path), "citi_airflow:/opt/airflow/dags/"]
result = subprocess.run(copy_cmd, capture_output=True, text=True)

if result.returncode != 0:
    print(result.stdout)
    print(result.stderr)
    raise RuntimeError("Failed to copy DAG file into Airflow container")

print("DAG file copied to citi_airflow container")
print("Waiting 35s for Airflow scheduler to pick up the DAG...")
time.sleep(35)

DAG file copied to citi_airflow container
Waiting 35s for Airflow scheduler to pick up the DAG...


Check the Airflow API to confirm the DAG is registered

In [6]:
dag_url = "http://localhost:8082/api/v1/dags/citi_alert_summary"
response = requests.get(dag_url, auth=("admin", "admin"), timeout=30)

print("Status code:", response.status_code)
if response.status_code == 401:
    print("Airflow API unauthorized; falling back to Docker CLI for DAG checks")
    USE_DOCKER_CLI = True
elif response.status_code == 404:
    print("DAG not found — wait another 30s and re-run this cell")
else:
    response.raise_for_status()
    payload = response.json()
    print("dag_id:", payload.get("dag_id"))
    print("schedule_interval:", payload.get("schedule_interval") or payload.get("schedule_interval_description") or payload.get("timetable_description"))
    print("is_paused:", payload.get("is_paused"))


Status code: 401
Airflow API unauthorized; falling back to Docker CLI for DAG checks


New DAGs start paused. Unpause then trigger a manual run.

In [7]:
if USE_DOCKER_CLI:
    import subprocess
    print("Unpausing and triggering DAG via Docker CLI...")
    subprocess.run(["docker", "exec", "citi_airflow", "airflow", "dags", "unpause", "citi_alert_summary"], check=True)
    logical_date = datetime.utcnow().replace(microsecond=0).isoformat() + "Z"
    subprocess.run(["docker", "exec", "citi_airflow", "airflow", "dags", "trigger", "citi_alert_summary", "-e", logical_date], check=True)
    print("DAG run triggered via CLI")
else:
    patch_url = "http://localhost:8082/api/v1/dags/citi_alert_summary"
    patch_response = requests.patch(
        patch_url,
        auth=("admin", "admin"),
        headers={"Content-Type": "application/json"},
        data=json.dumps({"is_paused": False}),
        timeout=30,
    )
    patch_response.raise_for_status()
    print("DAG unpaused")

    logical_date = datetime.utcnow().replace(microsecond=0).isoformat() + "Z"
    trigger_url = "http://localhost:8082/api/v1/dags/citi_alert_summary/dagRuns"
    trigger_response = requests.post(
        trigger_url,
        auth=("admin", "admin"),
        headers={"Content-Type": "application/json"},
        data=json.dumps({"logical_date": logical_date}),
        timeout=30,
    )
    trigger_response.raise_for_status()

    trigger_payload = trigger_response.json()
    print(f"DAG run triggered: {trigger_payload['dag_run_id']}")


Unpausing and triggering DAG via Docker CLI...


C:\Users\shareuser\AppData\Local\Temp\ipykernel_58516\3682099876.py:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  logical_date = datetime.utcnow().replace(microsecond=0).isoformat() + "Z"


DAG run triggered via CLI


Poll until the run succeeds or fails (max 60s)

In [8]:
if USE_DOCKER_CLI:
    print("Skipping API polling due to unauthorized API access. Use Airflow UI to verify run state.")
else:
    runs_url = "http://localhost:8082/api/v1/dags/citi_alert_summary/dagRuns"
    final_state = None

    for attempt in range(12):
        resp = requests.get(runs_url, auth=("admin", "admin"), timeout=30)
        resp.raise_for_status()
        payload = resp.json()
        dag_runs = payload.get("dag_runs", [])

        if not dag_runs:
            print(f"Attempt {attempt + 1}: no dag runs found yet")
            time.sleep(5)
            continue

        latest_run = sorted(
            dag_runs,
            key=lambda x: x.get("start_date") or x.get("logical_date") or "",
            reverse=True
        )[0]
        state = latest_run.get("state")
        print(f"Attempt {attempt + 1}: state={state}")

        if state in ("success", "failed"):
            final_state = state
            break

        time.sleep(5)

    print("Final outcome:", final_state if final_state else "timeout / still running")


Skipping API polling due to unauthorized API access. Use Airflow UI to verify run state.


The DAG wrote to alert_daily_summary — let's verify

In [9]:
try:
    conn = psycopg2.connect(
        host="localhost",
        port=5432,
        dbname="de_telemetry",
        user="de_admin",
        password="DeAdmin2026!",
    )
    cur = conn.cursor()
    cur.execute("SET search_path TO telemetry, public")
    cur.execute(
        '''
        SELECT report_date, severity, alert_count, created_at
        FROM alert_daily_summary
        ORDER BY report_date DESC, created_at DESC
        LIMIT 10
        '''
    )
    rows = cur.fetchall()

    for row in rows:
        print(row)

    cur.close()
    conn.close()
except Exception as e:
    print("Summary table not available yet. Wait for the DAG to finish and re-run this cell.")
    print(e)


Summary table not available yet. Wait for the DAG to finish and re-run this cell.
relation "alert_daily_summary" does not exist
LINE 3:         FROM alert_daily_summary
                     ^



## What Just Happened

- Authored a DAG in TaskFlow API
- Deployed it with `docker cp`
- Triggered it through the Airflow REST API
- Confirmed the output landed in Postgres

Citi tie-in: **This DAG runs every morning at 06:00. Ops teams get fresh alert counts without any manual intervention. Add a Slack notification task to make it production-ready.**

Next: **Run `airflow_concepts.md` for vocabulary, then Round 2 for production patterns.**